In [ ]:
import os
import sys
from pathlib import Path
import logging
import boto3

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s - %(message)s",
    force=True,
)
logging.getLogger("recon_validators").setLevel(logging.INFO)

print("Python:", sys.version)
print("Working directory:", Path.cwd())

Python: 3.12.9 | packaged by conda-forge | (main, Feb 14 2025, 08:00:06) [GCC 13.3.0]
Working directory: /mnt/custom-file-systems/s3/shared/suhail_workspace


In [2]:
import boto3

# ---- S3 artifact location ----
S3_BUCKET = "amazon-sagemaker-438465132548-us-east-1-242038a46fe7"
S3_KEY = "dzd_aow04z6eei0d47/6k5768rke0n2cn/dev/custom_packages/recon_validators-0.1.0-py3-none-any.whl"

# Local destination (visible in notebook working folder)
WHEEL_LOCAL_DIR = Path.cwd() / "python_wheels"
WHEEL_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
WHEEL_LOCAL_PATH = WHEEL_LOCAL_DIR / Path(S3_KEY).name

if WHEEL_LOCAL_PATH.exists():
    print("Wheel already downloaded, skipping:", WHEEL_LOCAL_PATH)
else:
    s3 = boto3.client("s3")
    s3.download_file(S3_BUCKET, S3_KEY, str(WHEEL_LOCAL_PATH))
    print("Downloaded wheel to:", WHEEL_LOCAL_PATH)

Wheel already downloaded, skipping: /mnt/custom-file-systems/s3/shared/suhail_workspace/python_wheels/recon_validators-0.1.0-py3-none-any.whl


In [3]:
# Install wheel in current kernel environment WITH dependencies
!{sys.executable} -m pip uninstall -y recon-validators recon_validators
!{sys.executable} -m pip install --no-cache-dir --force-reinstall {WHEEL_LOCAL_PATH}

Found existing installation: recon-validators 0.1.0
Uninstalling recon-validators-0.1.0:
  Successfully uninstalled recon-validators-0.1.0
Processing ./python_wheels/recon_validators-0.1.0-py3-none-any.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 318.7 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.7.0
    Uninstalling urllib3-2.7.0:
      Successfully uninstalled urllib3-2.7.0
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.1.4
    Uninstalling tenacity-9.1.4:
      Successfully uninstalled tenacity-9.1.4
  Attempting uninstall: six
    Found existing installation: six 1.17.0━━━━━━━━━━━━━━━━━━━━━━━  2/14 [six]
    Uninstalling six-1.17.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/14 [six]
      Successfully uninstalled six-1.17.0━━━━━━━━━━━━━━━━━━━━━━━━━  2/14 [six]
  Attempting uninstall: packaging━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/14 [six]
    Found existing installation: packaging 26.2━━━━━━━━━━━━━━━  2

In [4]:
# Simple and reliable import from downloaded wheel
import importlib
import sys

# Clear previously loaded package modules from current kernel
for mod_name in list(sys.modules.keys()):
    if mod_name == "recon_validators" or mod_name.startswith("recon_validators."):
        del sys.modules[mod_name]

# Ensure wheel path is first in import path
wheel_path = str(WHEEL_LOCAL_PATH)
if wheel_path in sys.path:
    sys.path.remove(wheel_path)
sys.path.insert(0, wheel_path)
importlib.invalidate_caches()

from recon_validators.connections import (
    JDBCImpalaConnection,
    AthenaConnection,
    ConnectionManager,
)
from recon_validators.runner import ValidationRunner
import recon_validators

print("✓ Imports loaded")
print("Loaded from:", recon_validators.__file__)

✓ Imports loaded
Loaded from: /mnt/custom-file-systems/s3/shared/suhail_workspace/python_wheels/recon_validators-0.1.0-py3-none-any.whl/recon_validators/__init__.py


In [5]:
# Download Impala JDBC jar into local ./jars directory (only once)
from pathlib import Path
import urllib.request

JAR_URL = "https://repository.cloudera.com/repository/cloudera-repos/Impala/ImpalaJDBC42/2.6.26.1031/ImpalaJDBC42-2.6.26.1031.jar"
JAR_NAME = "ImpalaJDBC42-2.6.26.1031.jar"
JARS_DIR = Path.cwd() / "jars"
JARS_DIR.mkdir(parents=True, exist_ok=True)

jar_path = JARS_DIR / JAR_NAME
if jar_path.exists():
    print(f"Jar already exists, skipping download: {jar_path}")
else:
    print(f"Downloading {JAR_NAME} to {jar_path}...")
    urllib.request.urlretrieve(JAR_URL, jar_path)
    print(f"Downloaded to: {jar_path}")

Jar already exists, skipping download: /mnt/custom-file-systems/s3/shared/suhail_workspace/jars/ImpalaJDBC42-2.6.26.1031.jar


In [6]:
# ---- JDBC jar handling ----
from pathlib import Path

JAR_NAME = "ImpalaJDBC42-2.6.26.1031.jar"
JAR_PATH = str(Path.cwd() / "jars" / JAR_NAME)

if not Path(JAR_PATH).exists():
    raise FileNotFoundError(
        f"Impala JDBC jar not found at {JAR_PATH}. Run the previous jar download cell first."
    )

print("Using JAR:", JAR_PATH)

Using JAR: /mnt/custom-file-systems/s3/shared/suhail_workspace/jars/ImpalaJDBC42-2.6.26.1031.jar


In [ ]:
# ---- Connection config ----
IMPALA_JDBC_URL = (
    "jdbc:impala://peanut-impala.cargill.com:21050;"
    "AuthMech=3;"
    "SSL=1;"
    "AllowSelfSignedCerts=1"
)
IMPALA_USERNAME = os.getenv("IMPALA_USERNAME", "ps696636@AP.CORP.CARGILL.COM")
IMPALA_PASSWORD = os.getenv("IMPALA_PASSWORD", "WeRock@123")

ATHENA_REGION = "us-east-1"
ATHENA_S3_STAGING_DIR = "s3://amazon-sagemaker-438465132548-us-east-1-242038a46fe7/dzd_aow04z6eei0d47/6k5768rke0n2cn/dev/sys/athena/"
ATHENA_WORKGROUP = "workgroup-6k5768rke0n2cn-4csydjgqkcyhpj"

In [8]:
conn_manager = ConnectionManager()

source_conn = JDBCImpalaConnection(
    jdbc_url=IMPALA_JDBC_URL,
    username=IMPALA_USERNAME,
    password=IMPALA_PASSWORD,
    jar_path=JAR_PATH,
)

target_conn = AthenaConnection(
    region_name=ATHENA_REGION,
    s3_staging_dir=ATHENA_S3_STAGING_DIR,
    workgroup=ATHENA_WORKGROUP,
)

conn_manager.set_source(source_conn)
conn_manager.set_target(target_conn)
print("✓ Connections initialized")

✓ Connections initialized


In [9]:
# Smoke tests
print("Source test:", conn_manager.source.execute("SELECT 1 AS ok"))
print("Target test:", conn_manager.target.execute("SELECT 1 AS ok"))

Source test: [{'ok': 1}]
Target test: [{'ok': 1}]


In [ ]:
# ---- Legacy Use Case Example ----
table_configs = [
    {
        "source_database": "prd_internal_tc1",
        "source_table": "afko",
        "target_database": "minerva_dev_src_sap_cdp_tc1_prd_raw_db",
        "target_table": "afko",
        "tc1": {"type": "count_check", "is_enabled": True, "tolerance_in_percent": 0.0},
        "tc2": {
            "type": "column_name_check",
            "is_enabled": True,
            "skip_columns": ["updated_at"],
        },
        "tc3": {"type": "column_datatype_check", "is_enabled": False},
        "tc4": {
            "type": "not_null_check",
            "is_enabled": True,
            "validation_list": ["aufnr", "xflag"],
        },
        "tc5": {
            "type": "unique_check",
            "is_enabled": True,
            "validation_list": ["aufnr", ["aufnr", "xflag"]],
        },
        "tc6": {"type": "length_check", "is_enabled": False},
    }
]

runner = ValidationRunner(
    source_connection=conn_manager.source,
    target_connection=conn_manager.target,
    executed_by=os.getenv("USER", "smus-user"),
)

results = runner.run(table_configs)
runner.print_summary(results)
out_file = runner.save_to_json(results, output_dir="validation_results")
print("Saved:", out_file)

Error: 

In [ ]:
# ---- Basic Recon Use Case Example ----
basic_table_configs = [
    {
        "recon_type": "basic",
        "source_database": "prd_internal_tc1",
        "target_database": "minerva_dev_src_sap_cdp_tc1_prd_raw_db",
        "tables_list": [
            "afko:afko(updated_at)",
        ],
    }
]

basic_runner = ValidationRunner(
    source_connection=conn_manager.source,
    target_connection=conn_manager.target,
    executed_by=os.getenv("USER", "smus-user"),
)

basic_results = basic_runner.run(basic_table_configs)
basic_runner.print_summary(basic_results)
basic_out_file = basic_runner.save_to_json(
    basic_results, output_dir="validation_results_basic"
)
print("Basic saved:", basic_out_file)

In [ ]:
# ---- Spark session discovery for spark_sql_check in SMUS/Jupyter ----
import os
import sys

print("Kernel executable:", sys.executable)
print("SMUS/Jupyter note: Spark is only available when this notebook uses a Spark-enabled runtime/kernel.")

spark_session = globals().get("spark")
spark_source = "global spark variable" if spark_session is not None else None

if spark_session is None:
    try:
        from pyspark.sql import SparkSession

        spark_session = SparkSession.getActiveSession()
        if spark_session is not None:
            spark_source = "active PySpark session"
        elif os.getenv("RECON_CREATE_LOCAL_SPARK", "false").lower() == "true":
            spark_session = SparkSession.builder.appName("recon_validators_sample").getOrCreate()
            spark_source = "local SparkSession.builder"
    except ImportError:
        print("PySpark is not installed in this kernel.")
        spark_session = None

spark_sql_enabled = spark_session is not None
if spark_sql_enabled:
    spark = spark_session
    logging.info("Spark session found from %s; spark_sql_check will run.", spark_source)
    print("✓ Spark session available from:", spark_source)
    print("Spark version:", spark_session.version)
else:
    logging.info(
        "No Spark session found in this SMUS/Jupyter kernel; spark_sql_check will be skipped. "
        "Use a Spark-enabled SMUS runtime/kernel, or set RECON_CREATE_LOCAL_SPARK=true only when local PySpark is configured."
    )
    print("Spark session not available; spark_sql_check will be skipped.")
    print("To run spark_sql_check in SMUS, switch this notebook to a Spark-enabled runtime/kernel and rerun this cell.")

In [ ]:
# ---- DQ Recon Use Case Example ----
spark_session = globals().get("spark")
spark_sql_enabled = spark_session is not None
if not spark_sql_enabled:
    logging.info(
        "No Spark session found; spark_sql_check will be skipped in this notebook run."
    )

dq_table_configs = [
    {
        "recon_type": "dq",
        "source_database": "prd_internal_tc1",
        "source_table": "afko",
        "target_database": "minerva_dev_src_sap_cdp_tc1_prd_raw_db",
        "target_table": "afko",
        "tc1": {"type": "count_check", "is_enabled": True, "tolerance_in_percent": 0.0},
        "tc2": {
            "type": "column_name_check",
            "is_enabled": True,
            "skip_columns": ["updated_at"],
        },
        "tc3": {
            "type": "column_datatype_check",
            "is_enabled": True,
            "skip_columns": ["updated_at"],
        },
        "tc4": {
            "type": "not_null_check",
            "is_enabled": True,
            "validation_list": ["aufnr", "xflag"],
        },
        "tc5": {
            "type": "unique_check",
            "is_enabled": True,
            "validation_list": ["aufnr", ["aufnr", "xflag"]],
        },
        "sql_1": {
            "type": "sql_check",
            "is_enabled": True,
            "source_query": """
                SELECT *
                FROM prd_internal_tc1.afko
                WHERE 1 = 1
                  AND gmein = 'KG'
                  AND plnme != 'KG'
            """,
            "target_query": """
                SELECT *
                FROM minerva_dev_src_sap_cdp_tc1_prd_raw_db.afko
                WHERE 1 = 1
                  AND gmein = 'KG'
                  AND plnme != 'KG'
            """,
            "max_sample_rows": 20,
        },
        "spark_sql_1": {
            "type": "spark_sql_check",
            "is_enabled": spark_sql_enabled,
            "source_query": """
                SELECT *
                FROM prd_internal_tc1.afko
                WHERE 1 = 1
                  AND gmein = 'KG'
                  AND plnme != 'KG'
            """,
            "target_query": """
                SELECT *
                FROM minerva_dev_src_sap_cdp_tc1_prd_raw_db.afko
                WHERE 1 = 1
                  AND gmein = 'KG'
                  AND plnme != 'KG'
            """,
            "max_sample_rows": 20,
        },
    }
]

dq_runner = ValidationRunner(
    source_connection=conn_manager.source,
    target_connection=conn_manager.target,
    executed_by=os.getenv("USER", "smus-user"),
    spark_session=spark_session,
)

dq_results = dq_runner.run(dq_table_configs)
dq_runner.print_summary(dq_results)
dq_out_file = dq_runner.save_to_json(dq_results, output_dir="validation_results_dq")
print("DQ saved:", dq_out_file)

In [11]:
# Cleanup
conn_manager.close_all()
print("✓ Connections closed")

✓ Connections closed
